# CNN for 52 Card + 1 Joker deck classification

In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import cv2
import matplotlib.pyplot as plt

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device is: {device}")

Device is: cuda


In [3]:
BATCH_SIZE = 32
transform = transforms.Compose([
    #transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.05),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

basic_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset = datasets.ImageFolder(root='../card-dataset/train', transform=transform)
test_dataset = datasets.ImageFolder(root='../card-dataset/test', transform=basic_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## Definición de la Red

In [6]:
import torch.nn as nn
import torch.nn.functional as F

CONV_LAYERS = [3, 64, 128, 128]
FC_LAYERS = [CONV_LAYERS[-1] * 29 * 29, 1024, 256, 53]

# MODEL -------------------------------------------------------
class CardNet(nn.Module):
    def __init__(self, fc_drop=0.3):
        super(CardNet, self).__init__()

        self.conv1 = nn.Conv2d(CONV_LAYERS[0], CONV_LAYERS[1], kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(CONV_LAYERS[1], CONV_LAYERS[2], kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(CONV_LAYERS[2], CONV_LAYERS[-1], kernel_size=3, stride=1, padding=1)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=1)

        # Dropout layers
        self.dropout_fc = nn.Dropout(p=fc_drop)

        self.fc1 = nn.Linear(FC_LAYERS[0], FC_LAYERS[1])
        self.fc2 = nn.Linear(FC_LAYERS[1], FC_LAYERS[2])
        self.fc3 = nn.Linear(FC_LAYERS[2], FC_LAYERS[-1])

    def forward(self, x):
        # CONVOLUTIONAL BLOCK
        x = F.relu(self.conv1(x))
        x = self.pool(x)

        x = F.relu(self.conv2(x))
        x = self.pool(x)

        x = F.relu(self.conv3(x))
        x = self.pool(x)

        x = x.view(-1, FC_LAYERS[0])  # RESIZE FOR FC BLOCK

        # FULLY CONNECTED
        x = F.relu(self.fc1(x))
        x = self.dropout_fc(x)     # Dropout after FC1

        x = F.relu(self.fc2(x))
        x = self.dropout_fc(x)     # Dropout after FC2

        return self.fc3(x)

 # Bucle de entrenamiento

In [11]:
import torch.optim as optim
from math import inf

model = CardNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# TRAINING -------------------------------------------------------
EPOCHS = 15
last_loss = inf
for epoch in range(EPOCHS):
    running_loss = 0.0
    if epoch == 15:
        for param_group in optimizer.param_groups:
            param_group['lr'] = 1e-4
            print("Switched LR to 1e-4")
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        running_loss += loss.item()
    if running_loss > last_loss:
        print(f"Stopped early at epoch {epoch+1}, with a loss of {running_loss/len(train_loader)}")
        break
    last_loss = running_loss
    torch.save(model.state_dict(), "best-train-model-weights.pth")

    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

OutOfMemoryError: CUDA out of memory. Tried to allocate 422.00 MiB. GPU 0 has a total capacity of 3.68 GiB of which 41.06 MiB is free. Including non-PyTorch memory, this process has 3.62 GiB memory in use. Of the allocated memory 2.82 GiB is allocated by PyTorch, and 731.73 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Bucle de evaluación

In [7]:
correct = 0
total = 0

model = CardNet().to(device)
model.load_state_dict(torch.load("cardNet-weights.pth"))
model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

precision = 100 * correct / total
print(f'Precisión del modelo en el conjunto de prueba: {precision}%')
#if precision >= 85.67:
    #torch.save(model.state_dict(), "cardNet-weights.pth")

torch.Size([32, 64, 113, 113])
torch.Size([32, 128, 57, 57])
torch.Size([32, 128, 29, 29])
torch.Size([32, 107648])
torch.Size([32, 64, 113, 113])
torch.Size([32, 128, 57, 57])
torch.Size([32, 128, 29, 29])
torch.Size([32, 107648])
torch.Size([32, 64, 113, 113])
torch.Size([32, 128, 57, 57])
torch.Size([32, 128, 29, 29])
torch.Size([32, 107648])
torch.Size([32, 64, 113, 113])
torch.Size([32, 128, 57, 57])
torch.Size([32, 128, 29, 29])
torch.Size([32, 107648])
torch.Size([32, 64, 113, 113])
torch.Size([32, 128, 57, 57])
torch.Size([32, 128, 29, 29])
torch.Size([32, 107648])
torch.Size([32, 64, 113, 113])
torch.Size([32, 128, 57, 57])
torch.Size([32, 128, 29, 29])
torch.Size([32, 107648])
torch.Size([32, 64, 113, 113])
torch.Size([32, 128, 57, 57])
torch.Size([32, 128, 29, 29])
torch.Size([32, 107648])
torch.Size([32, 64, 113, 113])
torch.Size([32, 128, 57, 57])
torch.Size([32, 128, 29, 29])
torch.Size([32, 107648])
torch.Size([9, 64, 113, 113])
torch.Size([9, 128, 57, 57])
torch.Size([9